In [1]:
import os
import numpy as np
import torch
from PIL import Image
from scipy.ndimage import sobel
import torchvision.transforms as transforms
import re

# Define CUDA as device.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Variables iniciales proporcionadas
matrix_name = "DeepestRemovedBlobs-every02-EGU"
folder_path = r"X:/PhD_Gotelli/Exp20/EGU24Code/DeepestRemovedBlobs-every02/imgs_filtered"
# Ruta de la carpeta de destino para las imágenes exportadas
export_folder_path = os.path.join(folder_path, "../blocks")
os.makedirs(export_folder_path, exist_ok=True)

In [2]:
# Detectar bordes usando el operador de Sobel
def detect_edges(tensor):
    tensor_squeezed = tensor.squeeze().cpu().numpy()
    sx = sobel(tensor_squeezed, axis=0, mode="constant")
    sy = sobel(tensor_squeezed, axis=1, mode="constant")
    edges = np.hypot(sx, sy)
    edges_tensor = torch.tensor(edges, device=device)
    return edges_tensor.unsqueeze(0)  # Añade de nuevo una dimensión de lote

# Convertir imagen a tensor y luego obtener sus bordes
def load_image_to_tensor_with_edges(image_path, device):
    transform = transforms.Compose(
        [
            transforms.Grayscale(),  # Convertir imagen a escala de grises
            transforms.ToTensor(),  # Convertir imagen a tensor de PyTorch
        ]
    )
    image = Image.open(image_path)
    image_tensor = transform(image).unsqueeze(0).to(device)
    edges_tensor = detect_edges(image_tensor)
    return edges_tensor

# Función MHD en PyTorch
def calculate_mhd(tensor1, tensor2):
    def tensor_to_points(tensor):
        points = torch.nonzero(tensor.squeeze() > 0.1, as_tuple=False).float()
        return points[::2]  # Selecciona cada dos puntos

    A = tensor_to_points(tensor1)
    B = tensor_to_points(tensor2)
    if A.size(1) != B.size(1):
        raise ValueError("Ambos grupos de puntos tienen diferentes dimensiones.")
    fhd = torch.mean(torch.min(torch.cdist(A, B), dim=1)[0])
    rhd = torch.mean(torch.min(torch.cdist(B, A), dim=1)[0])
    mhd = torch.max(fhd, rhd)
    return mhd.item()

# Obtener la lista de imágenes
images = sorted([f for f in os.listdir(folder_path) if f.endswith(".png")])

# Convertir todas las imágenes en tensores
tensors = [
    load_image_to_tensor_with_edges(os.path.join(folder_path, img), device)
    for img in images
]

# Antes del bucle, verificamos si existen fragmentos guardados
chunk_files = [
    f
    for f in os.listdir(export_folder_path)
    if f.startswith(f"{matrix_name}_chunk_") and f.endswith(".npy")
]

# Extraemos el índice máximo procesado
max_i = -1
pattern = re.compile(rf"{re.escape(matrix_name)}_chunk_(\d+)_(\d+)\.npy")

for file in chunk_files:
    match = pattern.match(file)
    if match:
        start_i = int(match.group(1))
        end_i = int(match.group(2))
        if end_i > max_i:
            max_i = end_i

# Determinamos desde dónde continuar
start_i = max_i + 1

if start_i >= len(images):
    print("Todas las filas han sido procesadas.")
    exit()

    
    
    
start_i = 16500
print("Comenzando en "+str(start_i))


# Cargamos la matriz de distancia si existe
distance_matrix_file = os.path.join(
    export_folder_path, f"{matrix_name}_distance_matrix.npy"
)
if os.path.exists(distance_matrix_file):
    distance_matrix = np.load(distance_matrix_file)
else:
    # Inicializamos la matriz con ceros
    distance_matrix = np.zeros((len(tensors), len(tensors)))

# Bucle principal
for i in range(start_i, len(images)):
    if i % 10 == 0:
        print(f"Procesando fila {i}")
    for j in range(i + 1, len(tensors)):
        mhd = calculate_mhd(tensors[i], tensors[j])
        distance_matrix[i, j] = mhd  # Solo llenamos la parte triangular superior

    # Guardamos cada 100 filas
    if (i % 100 == 99) or (i == len(images) - 1):
        chunk_start = i - 99 if i >= 99 else 0
        chunk = distance_matrix[chunk_start : i + 1, :]
        np.save(
            os.path.join(
                export_folder_path, f"{matrix_name}_chunk_{chunk_start}_{i}.npy"
            ),
            chunk,
        )
        # También guardamos la matriz de distancia completa
        # np.save(distance_matrix_file, distance_matrix)

print("Proceso completado.")

Todas las filas han sido procesadas.
Comenzando en 16500
Procesando fila 16500
Procesando fila 16510
Procesando fila 16520
Procesando fila 16530
Procesando fila 16540
Procesando fila 16550
Procesando fila 16560
Procesando fila 16570
Procesando fila 16580
Procesando fila 16590
Procesando fila 16600
Procesando fila 16610
Procesando fila 16620
Procesando fila 16630
Procesando fila 16640
Procesando fila 16650
Procesando fila 16660
Procesando fila 16670
Procesando fila 16680
Procesando fila 16690
Procesando fila 16700
Procesando fila 16710
Procesando fila 16720
Procesando fila 16730
Procesando fila 16740
Procesando fila 16750
Procesando fila 16760
Procesando fila 16770
Procesando fila 16780
Procesando fila 16790
Procesando fila 16800
Procesando fila 16810
Procesando fila 16820
Procesando fila 16830
Procesando fila 16840
Procesando fila 16850
Procesando fila 16860
Procesando fila 16870
Procesando fila 16880
Procesando fila 16890
Procesando fila 16900
Procesando fila 16910
Procesando fila 169

KeyboardInterrupt: 